In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm  # For progress bars

# Define paths and models
DATA_ROOT = '/home/mellie/supervised-vs-SSL'
PASST_INTERMEDIATE = os.path.join(DATA_ROOT, '/data/3.1_intermediate_features_passt')
BYOLA_INTERMEDIATE = os.path.join(DATA_ROOT, '/data/3.2_intermediate_features_byola')
PASST_FINAL = os.path.join(DATA_ROOT, '/data/4.1_final_features_passt')
BYOLA_FINAL = os.path.join(DATA_ROOT, '/data/4.2_final_features_byola')
OUTPUT_DIR = os.path.join(DATA_ROOT, '/supervised_vs_ssl/PCA_results')

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load labels
# Assuming you have a CSV with file_names, class_labels, and task_labels (0=music, 1=speech)
# Replace this with your actual labels file
"""
for byola it's ['Folk' 'Rock' 'unknown' 'Electronic' 'Spoken Weird' 'Disco' 'Post-Punk' 'Avant-Garde'] and ['go' 'seven']
for passt it's ['Hip-Hop|Nerdcore|Alternative Hip-Hop' 'Reggae - Dancehall' 'Latin America' 'Hip-Hop' 'New Age|Instrumental' 'Folk' 'Hip-Hop|Alternative Hip-Hop|Rap' 'Ambient' 'Pop' 'Rock|Noise-Rock' 'Glitch|Chill-out' 'Soundtrack' 'Singer-Songwriter' 'Noise' 'Soundtrack|Instrumental' 'Rock|Indie-Rock|Garage' 'Avant-Garde|Noise|Experimental' 'Techno|House|Dance' 'Experimental Pop' 'Balkan 'Avant-Garde|Noise|Experimental|Improv' 'Experimental|Electroacoustic' 'Rock' 'Psych-Folk|Freak-Folk' 'Afrobeat|African' 'Electronic|Dubstep' 'Experimental' 'Latin America|Polka' 'Field Recordings|Unclassifiable' 'Instrumental' 'Folk|Singer-Songwriter' 'Celtic' 'International|Europe|Flamenco|Latin|Spanish' 'Pop|Experimental Pop' 'Minimalism' 'Reggae - Dub' 'Electronic' 'Punk|Electro-Punk|Industrial' 'Noise|Experimental|Improv' 'Experimental|Electroacoustic|Sound Art' 'Psych-Folk' 'Hip-Hop|Rap' 'Rock|Garage|Post-Punk' 'Experimental|Minimalism' 'Electronic|Chiptune|Chip Music' 'International|Balkan' 'Rock|Garage|Surf' 'Punk|Post-Rock|Hardcore''Rock|Punk' 'Soundtrack|Ambient|Instrumental' 'International|Europe' 'Electronic|House|Drum & Bass' 'Europe' 'Psych-Folk|Singer-Songwriter' 'International' 'Noise|Experimental|Drone' 'Soundtrack|Ambient' 'Electronic|Ambient Electronic|Dubstep' 'Avant-Garde|Experimental|Unclassifiable' 'French' 'Improv' 'Europe|Romany (Gypsy)|Klezmer' 'Electronic|Techno|House' 'Rock|Punk|Indie-Rock' 'Metal|Indie-Rock|Garage' 'Afrobeat''Ambient Electronic|Chill-out' 'Pop|Synth Pop' 'Experimental|Drone|Sound Collage' 'Ambient Electronic|Dubstep' 'Punk|Garage' 'Trip-Hop|Downtempo' 'Chiptune|Chip Music' 'Avant-Garde|Experimental' 'Lo-Fi|Surf' 'Avant-Garde|Field Recordings|Experimental|Electroacoustic|Musique Concrete' 'Chill-out' 'Polka|Balkan|Europe' 'Hip-Hop|Alternative Hip-Hop' 'Punk|Post-Punk' 'Sound Poetry' 'Hip-Hop|Rap|Hip-Hop Beats''Electroacoustic' 'Electronic|Trip-Hop|Chill-out' 'Psych-Rock' 'Hardcore' 'Shoegaze' 'Dubstep' 'Dance' 'Hip-Hop Beats' 'Holiday|Christmas' 'Rock|Surf' 'Electronic|Ambient Electronic|Drum & Bass' 'Salsa' 'Chill-out|Dubstep' 'International|Celtic' 'Avant-Garde|Electroacoustic' 'Metal|Noise-Rock' 'Freak-Folk' 'Post-Rock|Progressive' 'New Age' 'Punk|Lo-Fi|Garage' 'Ambient Electronic' 'Post-Punk|Power-Pop' 'Alternative Hip-Hop|Rap|Hip-Hop Beats' 'Rap' 'Field Recordings''Experimental Pop|Synth Pop' 'Lo-Fi|Death-Metal|Grindcore' 'Rock|Metal|Loud-Rock' 'Chiptune' 'Rock|Indie-Rock' 'Experimental|Sound Collage|Sound Art' 'IDM|Drum & Bass|Jungle' 'Black-Metal' 'Rock|Indie-Rock|Progressive' 'Avant-Garde|Sound Collage' 'Metal' 'Middle East|Balkan|Romany (Gypsy)' 'Novelty|Kid-Friendly' 'Romany (Gypsy)' 'Electroacoustic|Sound Collage' 'Electronic|Chiptune' 'Krautrock|Psych-Rock|Space-Rock' 'Electronic|Dance' 'Electronic|House|Chill-out' 'Balkan|Europe' 'Electronic|Chip Music|Skweee' 'Punk|Electro-Punk' 'Garage|Power-Pop' 'Indie-Rock|Post-Punk|Goth' 'Lo-Fi|Indie-Rock|Post-Punk' 'Punk|Hardcore' 'Electronic|Ambient Electronic|Chill-out' 'Surf' 'Electronic|Glitch|Dubstep' 'Rock|Industrial' 'Electronic|Techno|Dance|Chill-out|Downtempo' 'Electronic|Drum & Bass|Jungle' 'Rock|Goth|Surf' 'Electronic|Techno|Bigbeat' 'Sound Collage|Musique Concrete|Sound Art' 'Folk|Psych-Folk|Singer-Songwriter' 'Noise-Rock|Psych-Rock' 'New Wave' 'Lo-Fi|Metal' 'Post-Rock' 'Asia-Far East' 'International|Middle East|Turkish' 'Indie-Rock|Progressive' 'Space-Rock' 'Indie-Rock' 'Audio Collage|Experimental' 'Electronic|Ambient Electronic|Downtempo' 'Alternative Hip-Hop' 'Trip-Hop|Dance|Chill-out|Bigbeat' 'Rock|Metal' 'Punk|Noise-Rock' 'Indie-Rock|Power-Pop' 'House|Dance|Chill-out' 'Synth Pop' 'Avant-Garde' 'Metal|Black-Metal' 'Noise|Experimental' 'Electronic|Techno|Dance' 'Techno|Dance' 'Indie-Rock|Post-Punk' 'Lo-Fi' 'Electronic|Dance|Skweee' 'Post-Rock|Metal' 'Ambient|New Age|Instrumental' 'International|Balkan|Romany (Gypsy)' 'Avant-Garde|Experimental|Sound Poetry' 'Reggae - Dub|Reggae - Dancehall' 'Rock|Post-Rock|Progressive' 'Electronic|Bigbeat' 'Ambient|Instrumental' 'Holiday' 'Death-Metal|Hardcore' 'Avant-Garde|Experimental|Electroacoustic' 'Electronic|Techno' 'Rock|Electro-Punk|Goth' 'Noise|Sound Collage' 'Psych-Rock|Garage' 'Avant-Garde|Experimental|Sound Collage' 'Tango' 'International|Middle East|Europe' 'Psych-Rock|Indie-Rock' 'Electronic|Skweee' 'Indie-Rock|Space-Rock' 'Techno|House' 'Free-Folk' 'House|Chip Music|Dubstep' 'Electronic|Chip Music|Dubstep' 'Indie-Rock|Space-Rock|Shoegaze' 'Post-Punk' 'Electronic|House|Chiptune' 'Electronic|IDM|Dance' 'Middle East|Balkan|Europe' 'Garage|Surf' 'International|Afrobeat' 'Noise|Improv' 'Rock|Punk|Rock Opera' 'Rock|Garage|Power-Pop' 'Experimental|Improv|Minimalism' 'Rock|Psych-Rock|Indie-Rock' 'Electronic|House|Dance' 'Rock|Psych-Rock|Garage' 'Drum & Bass|Jungle' 'Audio Collage' 'Psych-Folk|Free-Folk' 'Rock|Punk|Garage''Rock|Loud-Rock|Psych-Rock|Indie-Rock' 'Electronic|Trip-Hop' 'Ambient Electronic|Drum & Bass|Bigbeat' 'Indie-Rock|Garage|New Wave' 'Chiptune|Dance|Chip Music' 'Punk|Post-Punk|Shoegaze' 'Ambient Electronic|Trip-Hop|Chill-out' 'Field Recordings|Minimalism' 'Electronic|Techno|Drum & Bass' 'Electronic|House|IDM' 'Europe|Romany (Gypsy)' 'Unclassifiable' 'Punk|Metal|Loud-Rock' 'Avant-Garde|Experimental|Electroacoustic|Improv' 'Electro-Punk' 'Sound Collage' 'International|Romany (Gypsy)|Turkish' 'Latin' 'House|Dance' 'Rock|Indie-Rock|Post-Punk' 'Indie-Rock|Garage|Power-Pop' 'International|Balkan|Flamenco' 'Experimental|Drone|Minimalism' 'Novelty' 'House' 'Reggae - Dub|Cumbia' 'Electronic|Techno|Dubstep' 'Noise-Rock|Death-Metal|Hardcore|Black-Metal' 'Jungle' 'Pop|Experimental Pop|Synth Pop' 'Folk|Psych-Folk' 'Avant-Garde|Experimental|Unclassifiable|Improv|Sound Art' 'Avant-Garde|Experimental|Improv' 'Downtempo' 'African|Middle East' 'International|Latin America|Cumbia' 'Trip-Hop|Dubstep' 'Noise|Experimental|Unclassifiable' 'Experimental|Unclassifiable' 'Drone' 'International|Reggae - Dancehall']
and ['right' 'no' 'off' 'up' 'yes' 'down' 'cat' 'stop' 'dog' 'go' 'marvin' 'sheila' 'nine' 'on' 'eight' 'bird' 'four' 'two' 'seven' 'one' 'five' 'three' 'tree' 'happy' 'wow' 'six' 'house' 'left' 'zero' 'bed']
"""
# Load label CSVs and prepare task labels
# 1. Load PaSST predictions
passt_labels_df = pd.read_csv('/supervised_vs_ssl/passt_predictions.csv')
# 2. Load BYOL-A predictions
byola_labels_df = pd.read_csv('/supervised_vs_ssl/byola_predictions.csv')


# Create task labels (assuming 'actual_label' contains genre/speech category)
# Let's assume genres are music (0) and speech classes are (1)
# Modify this according to your actual label structure
def create_task_labels(df):
    # This is just an example - modify according to your actual label structure
    music_genres = ['rock', 'pop', 'jazz', 'classical', 'hiphop', 'electronic', 'country', 'folk']
    speech_types = ['command', 'conversation', 'lecture', 'news', 'podcast', 'audiobook']
    
    # Create binary task labels (0=music, 1=speech)
    task_labels = []
    for label in df['actual_label']:
        if any(genre in label.lower() for genre in music_genres):
            task_labels.append(0)  # Music
        elif any(speech in label.lower() for speech in speech_types):
            task_labels.append(1)  # Speech
        else:
            print(f"Warning: Unknown label type: {label}")
            task_labels.append(-1)  # Unknown
            
    return np.array(task_labels)

# Create task labels
passt_task_labels = create_task_labels(passt_labels_df)
byola_task_labels = create_task_labels(byola_labels_df)

# Get file names (without extensions)
passt_file_names = [os.path.splitext(f)[0] for f in passt_labels_df['file']]
byola_file_names = [os.path.splitext(f)[0] for f in byola_labels_df['file']]


def prepare_layer_features(feature_dir, file_names):
    """
    Load and prepare features for all files in a layer directory
    
    Args:
        feature_dir: Path to directory containing .npy files
        file_names: List of file names to process (should match the .npy files without extension)
        
    Returns:
        np.ndarray: Flattened features array with shape (n_samples, n_features)
    """
    all_features = []
    
    for fname in tqdm(file_names, desc=f"Processing {os.path.basename(feature_dir)}"):
        feature_path = os.path.join(feature_dir, f"{fname}.npy")
        if not os.path.exists(feature_path):
            print(f"Warning: {feature_path} not found, skipping...")
            continue
            
        features = np.load(feature_path)
        
        # Get the number of samples (first dimension)
        n_samples = features.shape[0] if features.ndim > 1 else 1
        
        # Flatten everything after the first dimension
        if features.ndim > 2:
            features_flat = features.reshape(n_samples, -1)
        elif features.ndim == 2:
            features_flat = features  # Already in right format
        else:
            features_flat = features.reshape(1, -1)
            
        all_features.append(features_flat)
        
    return np.vstack(all_features)

def process_model_layers(model_root, output_subdir, file_names, class_labels, task_labels):
    """
    Process all layers for a model and save PCA results
    
    Args:
        model_root: Root directory for model's intermediate features
        output_subdir: Subdirectory name for output
        file_names: List of file names
        class_labels: Original class labels
        task_labels: Binary task labels (0=music, 1=speech)
        
    Returns:
        dict: Dictionary with layer names and PCA results
    """
    # Create output directory
    output_dir = os.path.join(OUTPUT_DIR, output_subdir)
    os.makedirs(output_dir, exist_ok=True)
    
    # Get all layer directories
    layer_dirs = get_layer_dirs(model_root)
    
    # Store PCA results for each layer
    layer_pca_results = {}
    layer_info = []
    
    for layer_name in layer_dirs:
        layer_dir = os.path.join(model_root, layer_name)
        
        # Load and flatten features
        print(f"Processing layer: {layer_name}")
        features = prepare_layer_features(layer_dir, file_names)
        
        # Save flattened features
        flattened_dir = os.path.join(output_dir, f"{layer_name}_flattened")
        os.makedirs(flattened_dir, exist_ok=True)
        np.save(os.path.join(flattened_dir, "features.npy"), features)
        
        # Standardize features
        scaler = StandardScaler()
        features_scaled = scaler.fit_transform(features)
        
        # Apply PCA
        pca = PCA(n_components=2)
        features_pca = pca.fit_transform(features_scaled)
        
        # Save PCA results
        pca_dir = os.path.join(output_dir, f"{layer_name}_pca")
        os.makedirs(pca_dir, exist_ok=True)
        np.save(os.path.join(pca_dir, "pca_components.npy"), features_pca)
        
        # Save explained variance
        explained_variance = pca.explained_variance_ratio_
        np.save(os.path.join(pca_dir, "explained_variance.npy"), explained_variance)
        
        # Store in dictionary for visualization
        layer_pca_results[layer_name] = features_pca
        
        # Store layer info for CSV
        layer_info.append({
            'layer': layer_name,
            'n_samples': features.shape[0],
            'n_features_original': features.shape[1],
            'explained_variance_pc1': explained_variance[0],
            'explained_variance_pc2': explained_variance[1],
            'total_explained_variance': sum(explained_variance)
        })
    
    # Save layer info to CSV
    pd.DataFrame(layer_info).to_csv(
        os.path.join(output_dir, "layer_info.csv"), index=False
    )
    
    # Also save the labels for future use
    labels_df = pd.DataFrame({
        'file_name': file_names,
        'class_label': class_labels,
        'task_label': task_labels
    })
    labels_df.to_csv(os.path.join(output_dir, "labels.csv"), index=False)
    
    return layer_pca_results

def process_final_features(final_dir, output_subdir, file_names, class_labels, task_labels):
    """Process final embeddings and save PCA results"""
    # Create output directory
    output_dir = os.path.join(OUTPUT_DIR, output_subdir)
    os.makedirs(output_dir, exist_ok=True)
    
    # Load and flatten features
    print(f"Processing final embeddings from: {final_dir}")
    features = prepare_layer_features(final_dir, file_names)
    
    # Save flattened features
    np.save(os.path.join(output_dir, "features_flattened.npy"), features)
    
    # Standardize features
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)
    
    # Apply PCA
    pca = PCA(n_components=2)
    features_pca = pca.fit_transform(features_scaled)
    
    # Save PCA results
    np.save(os.path.join(output_dir, "pca_components.npy"), features_pca)
    
    # Save explained variance
    explained_variance = pca.explained_variance_ratio_
    np.save(os.path.join(output_dir, "explained_variance.npy"), explained_variance)
    
    # Save info to CSV
    info_df = pd.DataFrame([{
        'layer': 'final_embedding',
        'n_samples': features.shape[0],
        'n_features_original': features.shape[1],
        'explained_variance_pc1': explained_variance[0],
        'explained_variance_pc2': explained_variance[1],
        'total_explained_variance': sum(explained_variance)
    }])
    info_df.to_csv(os.path.join(output_dir, "info.csv"), index=False)
    
    return features_pca

def plot_layer_progression(features_dict, task_labels, class_labels, model_name, output_dir):
    """
    Plot PCA visualization for each layer in a grid
    
    Args:
        features_dict: Dictionary with layer names as keys and PCA components as values
        task_labels: Binary labels (0=music, 1=speech)
        class_labels: Original detailed class labels
        model_name: Name of the model ('PaSST' or 'BYOL-A')
        output_dir: Directory to save the visualization
    """
    n_layers = len(features_dict)
    n_cols = 3
    n_rows = (n_layers + n_cols - 1) // n_cols
    
    plt.figure(figsize=(n_cols * 6, n_rows * 5))
    
    for i, (layer_name, features_pca) in enumerate(features_dict.items()):
        # Create subplot
        plt.subplot(n_rows, n_cols, i+1)
        
        # Plot music points (red)
        plt.scatter(features_pca[task_labels == 0, 0], 
                   features_pca[task_labels == 0, 1],
                   c='red', alpha=0.7, s=30, label='Music')
        
        # Plot speech points (blue)
        plt.scatter(features_pca[task_labels == 1, 0], 
                   features_pca[task_labels == 1, 1],
                   c='blue', alpha=0.7, s=30, label='Speech')
        
        plt.title(f'Layer: {layer_name}')
        
        # Assuming you have pca.explained_variance_ratio_ values stored somewhere
        # If not, you can comment these lines out or modify them
        pc1_variance = 0  # Replace with actual value if available
        pc2_variance = 0  # Replace with actual value if available
        
        plt.xlabel(f'PC1 ({pc1_variance*100:.1f}%)')
        plt.ylabel(f'PC2 ({pc2_variance*100:.1f}%)')
        
        if i == 0:
            plt.legend()
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f'{model_name}_layer_progression.png'), dpi=300)
    plt.show()

# Process PaSST layers
print("Processing PaSST layers...")
passt_pca_results = process_model_layers(
    PASST_INTERMEDIATE, 
    'passt_layers', 
    file_names, 
    class_labels, 
    task_labels
)

# Process BYOLA layers
print("Processing BYOL-A layers...")
byola_pca_results = process_model_layers(
    BYOLA_INTERMEDIATE, 
    'byola_layers', 
    file_names, 
    class_labels, 
    task_labels
)

# Process final embeddings
print("Processing PaSST final embeddings...")
passt_final_pca = process_final_features(
    PASST_FINAL, 
    'passt_final', 
    file_names, 
    class_labels, 
    task_labels
)

print("Processing BYOL-A final embeddings...")
byola_final_pca = process_final_features(
    BYOLA_FINAL, 
    'byola_final', 
    file_names, 
    class_labels, 
    task_labels
)

# Add final embeddings to the layer results for visualization
passt_pca_results['final_embedding'] = passt_final_pca
byola_pca_results['final_embedding'] = byola_final_pca

# Create visualizations
print("Creating visualizations...")
plot_layer_progression(
    passt_pca_results, 
    task_labels, 
    class_labels, 
    'PaSST',
    OUTPUT_DIR
)

plot_layer_progression(
    byola_pca_results, 
    task_labels, 
    class_labels, 
    'BYOL-A',
    OUTPUT_DIR
)

print("PCA analysis completed and saved to:", OUTPUT_DIR)

In [ ]:
import numpy as np
# load features
features = np.load('layer_X_features.npy')

# flatten dimensions to 1D
def prepare_layer_features(feature_file, layer_name):
    features = np.load(feature_file)
    
    # Get the number of samples (first dimension)
    n_samples = features.shape[0] if features.ndim > 1 else 1
    
    # Flatten everything after the first dimension
    if features.ndim > 2:
        features_flat = features.reshape(n_samples, -1)
    elif features.ndim == 2:
        features_flat = features  # Already in right format
    else:
        features_flat = features.reshape(1, -1)
        
    return features_flat

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# normalize scale 
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features_flat)

# apply PCA
pca = PCA(n_components=2)  # or 3 for 3D
features_pca = pca.fit_transform(features_scaled)

In [ ]:
# visualize PCA with 2D scatter plot
import matplotlib.pyplot as plt

def plot_layer_progression(features_dict, task_labels, class_labels, model_name):
    """
    Plot PCA visualization for each layer in a grid
    
    Args:
        features_dict: Dictionary with layer names as keys and flattened features as values
        task_labels: Binary labels (0=music, 1=speech)
        class_labels: Original detailed class labels
        model_name: Name of the model ('PaSST' or 'BYOL-A')
    """
    n_layers = len(features_dict)
    n_cols = 3
    n_rows = (n_layers + n_cols - 1) // n_cols
    
    plt.figure(figsize=(n_cols * 6, n_rows * 5))
    
    for i, (layer_name, features) in enumerate(features_dict.items()):
        # Standardize and apply PCA
        scaler = StandardScaler()
        features_scaled = scaler.fit_transform(features)
        pca = PCA(n_components=2)
        features_pca = pca.fit_transform(features_scaled)
        
        # Create subplot
        plt.subplot(n_rows, n_cols, i+1)
        
        # Plot music points (red)
        plt.scatter(features_pca[task_labels == 0, 0], 
                   features_pca[task_labels == 0, 1],
                   c='red', alpha=0.7, s=30, label='Music')
        
        # Plot speech points (blue)
        plt.scatter(features_pca[task_labels == 1, 0], 
                   features_pca[task_labels == 1, 1],
                   c='blue', alpha=0.7, s=30, label='Speech')
        
        plt.title(f'Layer: {layer_name}')
        plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
        plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
        
        if i == 0:
            plt.legend()
    
    plt.tight_layout()
    plt.savefig(f'{model_name}_layer_progression.png', dpi=300)
    plt.show()

In [ ]:
# task specialization line plot showing where separation occurs
def plot_task_specialization(anova_results, model_name):
    """Plot how task separation evolves across layers"""
    plt.figure(figsize=(10, 6))
    
    # Plot F-values across layers
    plt.plot(anova_results['layer'], anova_results['task_f_value'], 
             'o-', linewidth=2, markersize=8, label='Task Separation')
    
    # Mark the layer with maximum separation
    max_idx = anova_results['task_f_value'].argmax()
    max_layer = anova_results.iloc[max_idx]['layer']
    max_f = anova_results.iloc[max_idx]['task_f_value']
    
    plt.annotate(f'Maximum separation\nat layer {max_layer}',
                xy=(max_layer, max_f), xytext=(max_layer, max_f*0.8),
                arrowprops=dict(arrowstyle='->'))
    
    plt.title(f'{model_name}: Task Specialization Across Layers')
    plt.xlabel('Layer')
    plt.ylabel('F-statistic (higher = better separation)')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(f'{model_name}_task_specialization.png', dpi=300)
    plt.show()